<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Experiments as a team habit

**[Experiments as a team habit](https://learning.nextia-ai.com/courses/experiments/m08/experiments-as-a-team-habit/)** · Experiments and Causal Impact: Did Our Change Work? · Module 8, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** simulate a year of tests for a team, count the false wins among the winners, measure how much real winners exaggerate (the winner's curse), and see what "testing until something wins" does.

| | |
|---|---|
| **Time** | About 25 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. The tests are simulated in the notebook. |
| **You should know** | Power and the minimum detectable effect, from [Sample size and power](https://learning.nextia-ai.com/courses/experiments/m02/sample-size-and-power/). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m08/experiments-as-a-team-habit/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M08-L04-experiments-as-a-team-habit/experiments-as-a-team-habit-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 "scipy==1.15.3; python_version < '3.14'" "scipy==1.18.1; python_version >= '3.14'" statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: the project folder

This notebook makes its own data with small simulations, so it downloads nothing. Run the next cell. It makes a folder `larkfield-experiments` and works in it.

You should see `Ready: 0 files in data/.`

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A year of 1,000 tests

Each test gives an estimate: the true effect plus noise with standard error `SE`. We choose `SE` so that the minimum detectable effect (MDE) is 1 with 80% power: SE = 1 / (1.96 + 0.84). 200 ideas really work, with an effect equal to the MDE; 800 do nothing. These numbers are an example, not a measurement.

In [ ]:
rng = np.random.default_rng(1)
SE = 1 / (1.96 + 0.84)
true = np.r_[np.ones(200), np.zeros(800)]
estimate = true + rng.normal(0, SE, true.size)
win = estimate > 1.96 * SE          # a win: the interval is above 0
print(f"standard error {SE:.3f}; a win needs an estimate above {1.96 * SE:.3f}")

> **Check.** You should see `standard error 0.357; a win needs an estimate above 0.700`.

## 2. How many wins are false?

In [ ]:
real_wins = int((win & (true > 0)).sum())
false_wins = int((win & (true == 0)).sum())
print(f"real wins {real_wins}, false wins {false_wins}, share false {false_wins / (real_wins + false_wins):.1%}")

> **Check.** You should see `real wins 158, false wins 16, share false 9.2%`. The lesson's tree, with exact expected numbers, has 160 and 20: about 1 in 9.

## 3. The winner's curse

In [ ]:
print(f"real winners: mean estimate {estimate[win & (true > 0)].mean():.2f}, true effect 1.00")
print(f"all good ideas: mean estimate {estimate[true > 0].mean():.2f}")

> **Check.** You should see `real winners: mean estimate 1.10, true effect 1.00`. Over all 200 good ideas, the mean is close to 1: the tests are fair. Only the winners, chosen because their luck was good, are too high.

## 4. Your turn: testing until something wins

A team tests ideas that do nothing, one after another, and stops at the first win, after at most 10 tests. We simulate 10,000 such teams.

> **Your turn.** Simulate `TEAMS` teams with 10 tests of nothing each (estimates from `rng.normal(0, SE, size=(TEAMS, 10))`). A team finds a false win if any of its 10 estimates is above `1.96 * SE`. Put the share of teams with a false win in `false_win_share`, rounded to 3 decimals.

In [ ]:
TEAMS = 10_000
false_win_share = ...  # the share of teams with at least one false win in 10 tests, rounded to 3 decimals

In [ ]:
expect('false_win_share', false_win_share, 0.224, tolerance=0.02)

About one team in five finds a "winner" among ten ideas that do nothing. A test log that counts every try makes this visible.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/experiments/m08/experiments-as-a-team-habit/) has the test log (`xp log`), the review before launch, and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Did the mailing work, and for whom?](https://learning.nextia-ai.com/courses/experiments/m09/did-the-mailing-work-and-for-whom/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m08/experiments-as-a-team-habit/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.